In [0]:
%run ./00_config

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType
from datetime import datetime, timezone, date
import json

spark = SparkSession.getActiveSession()

# =========================
# CUSTOM JSON ENCODER
# =========================

class SafeEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (datetime, date)):
            return obj.isoformat()
        return str(obj)  # fallback for any other non-serializable type


# =========================
# NORMALIZATION
# =========================

def normalize(records):
    cleaned = []
    for r in records:
        clean = {}
        for k, v in r.items():
            if v is False:
                clean[k] = None
            elif isinstance(v, (datetime, date)):
                clean[k] = v.strftime("%Y-%m-%d %H:%M:%S")
            elif isinstance(v, dict):
                clean[k] = json.dumps(v, cls=SafeEncoder) if v else None  # ← use SafeEncoder
            elif isinstance(v, list):
                if len(v) == 2 and isinstance(v[0], int):
                    clean[k] = str(v[0])
                elif v:
                    clean[k] = json.dumps(v, cls=SafeEncoder)  # ← use SafeEncoder
                else:
                    clean[k] = None
            else:
                clean[k] = str(v) if v is not None else None
        cleaned.append(clean)
    return cleaned


# =========================
# LOAD TO DELTA (BRONZE)
# =========================

def load_to_bronze(records, table_name):
    if not records:
        return

    delta_table = table_name.replace(".", "_")
    full_table = f"{CATALOG}.{BRONZE_SCHEMA}.{delta_table}"

    existing_columns = [f.name for f in spark.table(full_table).schema.fields]

    now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

    filtered_records = []
    for r in records:
        r["dwh_loaded_at"] = now
        r["dwh_source_db"] = ODOO_DB
        filtered = {k: v for k, v in r.items() if k in existing_columns}
        filtered_records.append(filtered)

    schema = StructType([StructField(col, StringType(), True) for col in existing_columns])
    df = spark.createDataFrame(filtered_records, schema=schema)

    # Use MERGE to avoid duplicates — upsert on id
    df.createOrReplaceTempView("bronze_updates")

    spark.sql(f"""
        MERGE INTO {full_table} t
        USING bronze_updates s
        ON t.id = s.id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)